# Orivox - Separador de Pistas (gratuito)

Separa sua musica em pistas individuais e gera extras para o Orivox.

### Pistas que voce recebe
Voz principal - Backing vocals - Bateria - Baixo - Guitarra - Piano - Outros

### Extras (opcionais)
Mapa de batidas (metronomo) - Cifra dos acordes - Partitura da voz

### Como usar
1. Menu **Ambiente de execucao > Alterar tipo de ambiente > T4 GPU** e salve.
2. Rode as celulas na ordem, clicando no play de cada uma.
3. Na celula 2, envie sua musica quando o botao aparecer.
4. A celula 5 baixa o ZIP com as pistas. As celulas 6+ sao extras opcionais.

*IMPORTANTE: as pistas (celulas 1 a 5) rodam primeiro e ficam prontas. Os extras
(6 em diante) instalam ferramentas mais pesadas so quando voce os roda, para NAO
atrapalhar a separacao das pistas.*

## Celula 1 - Instalar as ferramentas da separacao (rode uma vez por sessao)

In [ ]:
%%capture
# So o essencial para SEPARAR as pistas - sem pacotes pesados que dao conflito.
# Os extras (partitura, acordes) instalam suas ferramentas depois, nas celulas 6+.
!pip install -U demucs
!pip install "audio-separator[cpu]"


In [ ]:
print('Ferramentas da separacao instaladas! Siga para a celula 2.')

## Celula 2 - Enviar sua musica (MP3, WAV, M4A...)

In [ ]:
from google.colab import files
import os, shutil

# Aviso amigavel sobre a GPU (nao impede, so avisa).
try:
    import torch
    if torch.cuda.is_available():
        print('GPU ligada:', torch.cuda.get_device_name(0), '- otimo, vai ser rapido.')
    else:
        print('ATENCAO: GPU nao esta ligada. Funciona, mas mais devagar.')
        print('Para ligar: Ambiente de execucao > Alterar tipo > T4 GPU > Salvar. Depois rode de novo.')
except Exception:
    pass

if os.path.exists('/content/entrada'):
    shutil.rmtree('/content/entrada')
os.makedirs('/content/entrada', exist_ok=True)

print('\nClique em "Escolher arquivos" e selecione sua musica:')
enviados = files.upload()

for nome in enviados:
    shutil.move(nome, f'/content/entrada/{nome}')
    print(f'Recebido: {nome}')
arquivo = list(enviados.keys())[0]

## Celula 3 - Separar os instrumentos e a voz (Demucs, 6 pistas)

- `htdemucs_6s` gera 6 pistas: voz, bateria, baixo, guitarra, piano e outros

In [ ]:
MODO = 'htdemucs_6s'

import subprocess, os, shutil, glob
if os.path.exists('/content/saida'):
    shutil.rmtree('/content/saida')
entrada = f'/content/entrada/{arquivo}'
print(f'Separando "{arquivo}" em 6 pistas... aguarde (pode levar alguns minutos).')

proc = subprocess.run(
    ['demucs', '-n', MODO, '--mp3', '--mp3-bitrate', '320', '-o', '/content/saida', entrada],
    capture_output=True, text=True
)

# VERIFICACAO CRITICA: confere se as 6 pistas foram MESMO geradas.
# Se o demucs falhar, TRAVA aqui com dica clara - em vez de seguir e sobrar so a voz.
pistas = glob.glob('/content/saida/*/*/*.mp3')
esperadas = {'drums','bass','other','vocals','guitar','piano'}
achadas = set(os.path.splitext(os.path.basename(p))[0] for p in pistas)

if proc.returncode != 0 or not esperadas.issubset(achadas):
    print('=== A separacao em 6 pistas NAO foi concluida. ===')
    print('Pistas geradas:', sorted(achadas) if achadas else 'NENHUMA')
    print('Faltaram:', sorted(esperadas - achadas))
    print()
    print('--- Detalhes (ultimas linhas) ---')
    print((proc.stderr or proc.stdout or '')[-2000:])
    print()
    print('DICAS: 1) Ligue a GPU (Ambiente de execucao > T4 GPU) e rode de novo.')
    print('       2) Se faltou memoria, teste com uma musica mais curta.')
    print('       3) Se rodou algum extra (celula 6+) antes, reinicie o ambiente')
    print('          (Ambiente de execucao > Reiniciar) e rode da celula 1 de novo.')
    raise SystemExit('Separacao incompleta - veja as dicas acima.')

print('Pronto! As 6 pistas foram geradas:')
for p in sorted(pistas):
    print('  -', os.path.basename(p))

## Celula 4 - Dividir a voz em PRINCIPAL e BACKING VOCALS

Pega a pista de voz da celula 3 e isola a voz principal dos vocais de apoio.

In [ ]:
import glob, os, shutil
from audio_separator.separator import Separator

candidatos = glob.glob('/content/saida/*/*/vocals.*')
if not candidatos:
    raise SystemExit('Pista de voz nao encontrada. Rode a celula 3 primeiro (ela precisa terminar sem erro).')
voz_completa = candidatos[0]
print('Pista de voz localizada:', os.path.basename(voz_completa))

if os.path.exists('/content/vozes'):
    shutil.rmtree('/content/vozes')
os.makedirs('/content/vozes', exist_ok=True)

print('Separando voz principal dos backing vocals... aguarde.')
sep = Separator(output_dir='/content/vozes', output_format='mp3')
sep.load_model(model_filename='5_HP-Karaoke-UVR.pth')
saidas = sep.separate(voz_completa)

print('Pronto! Arquivos de voz gerados:')
for s in saidas:
    print('  -', os.path.basename(s))

## Celula 5 - Baixar as pistas (o essencial ja fica pronto aqui!)

Junta as 7 pistas e baixa o ZIP. **Se voce so quer as pistas, pode parar aqui.**
As celulas seguintes (6, 7, 8) sao extras opcionais.

In [ ]:
import shutil, os, glob
from google.colab import files

try:
    arquivo
except NameError:
    arquivo = os.path.basename(glob.glob('/content/entrada/*')[0])
base = os.path.splitext(arquivo)[0]
final = f'/content/{base}_Orivox'
if os.path.exists(final):
    shutil.rmtree(final)
os.makedirs(final, exist_ok=True)

# 1) Pistas do Demucs (a voz completa entra como referencia).
nomes_pt = {'drums':'Bateria','bass':'Baixo','other':'Outros','guitar':'Guitarra','piano':'Piano'}
pastas = [p for p in glob.glob('/content/saida/*/*') if os.path.isdir(p)]
if pastas:
    for f in glob.glob(f'{pastas[0]}/*'):
        raiz = os.path.splitext(os.path.basename(f))[0]
        ext = os.path.splitext(f)[1]
        if raiz == 'vocals':
            shutil.copy(f, f'{final}/Vozes juntas (referencia) - {base}{ext}')
        elif raiz in nomes_pt:
            shutil.copy(f, f'{final}/{nomes_pt[raiz]} - {base}{ext}')

# 2) Voz principal e backing vocals.
for f in glob.glob('/content/vozes/*'):
    nome = os.path.basename(f); ext = os.path.splitext(f)[1]
    if '(Vocals)' in nome:
        shutil.copy(f, f'{final}/Voz principal - {base}{ext}')
    elif '(Instrumental)' in nome:
        shutil.copy(f, f'{final}/Backing vocals - {base}{ext}')

# Confere quantas pistas de fato entraram
pistas_final = [f for f in os.listdir(final) if os.path.isfile(os.path.join(final,f))]
print(f'Pistas reunidas ({len(pistas_final)}):')
for f in sorted(pistas_final):
    print('  -', f)

if len(pistas_final) < 6:
    print('\nATENCAO: esperava 7 pistas mas achei', len(pistas_final), '.')
    print('Confira se as celulas 3 e 4 terminaram sem erro.')

zipado = shutil.make_archive(f'/content/{base}_Orivox_pistas', 'zip', final)
files.download(zipado)
print('\nDownload das pistas iniciado! Importe no console multipista do Orivox.')

## Celula 6 - EXTRA: Mapa de batidas (para o metronomo do Orivox)

Gera um `_batidas.json` com cada batida e tempo forte da musica. Importe no
Metronomo Inteligente do Orivox para o clique seguir a musica com precisao.
*Opcional - so rode se quiser o mapa.*

In [ ]:
# Instala o librosa SO AGORA (isolado), para nao atrapalhar a separacao das pistas.
import subprocess
subprocess.run(['pip','install','-q','librosa','soundfile'], check=False)

import json, os, glob
import numpy as np
from google.colab import files

entrada_lista = glob.glob('/content/entrada/*')
if not entrada_lista:
    raise SystemExit('Envie a musica na celula 2 primeiro.')
caminho = entrada_lista[0]
base = os.path.splitext(os.path.basename(caminho))[0]

import librosa
print('Analisando as batidas (librosa)... 1 a 2 minutos.')
y, sr = librosa.load(caminho, sr=22050, mono=True)
tempo, frames = librosa.beat.beat_track(y=y, sr=sr, trim=False)
beats = librosa.frames_to_time(frames, sr=sr).tolist()
env = librosa.onset.onset_strength(y=y, sr=sr)
forca = [float(env[min(f, len(env)-1)]) for f in frames]
melhor_o, melhor_s = 0, -1
for o in range(4):
    s = sum(forca[o::4])
    if s > melhor_s: melhor_s, melhor_o = s, o
numeros = [((i - melhor_o) % 4) + 1 for i in range(len(beats))]

iv = np.diff(beats)
bpm = round(float(60/np.median(iv)), 1) if len(iv) else 0
mapa = {'origem':'librosa', 'bpm':bpm, 'beats':[round(b,4) for b in beats], 'beat_numbers':numeros}
os.makedirs('/content/extras', exist_ok=True)
saida = f'/content/extras/{base}_batidas.json'
json.dump(mapa, open(saida,'w'))
print(f'Mapa gerado: {len(beats)} batidas, ~{bpm} BPM.')
files.download(saida)   # baixa SO o mapa (nao a pasta toda)
print('Importe no Metronomo Inteligente do Orivox.')

## Celula 7 - EXTRA: Partitura de referencia da voz (LIMPA)

Transcreve a voz principal em partitura com FIGURAS LIMPAS (estilo profissional):
detecta o BPM real e o tom (armadura de clave), extrai a melodia, consolida a
fragmentacao de vibrato, corrige as notas para dentro do tom e escreve figuras
validas (semicolcheia, colcheia, seminima, pontuadas) com ligaduras so quando
necessario. Gera MusicXML (MuseScore/Encore) e MIDI.

*As ALTURAS sao uma referencia (a fundamental da voz); voce ajusta na edicao.
As FIGURAS e o ritmo saem limpos, prontos para revisao.*

In [ ]:
# Partitura de referencia da voz - versao LIMPA (figuras profissionais).
# Cadeia: motor de BPM do Orivox + deteccao de tom + pyin + consolidacao +
# correcao pela escala + figuras limpas + divisao por compasso + armadura de clave.
import subprocess
subprocess.run(['pip','install','-q','librosa','soundfile','music21'], check=False)

import glob, os
import numpy as np
import librosa
from google.colab import files

NOTAS=['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

# ---- MOTOR DE BPM DO ORIVOX (detecta o BPM real, ex: 74 nao 152) ----
def detectar_bpm_orivox(y, sr):
    if sr > 22050:
        f=int(round(sr/22050)); y=y[::f]; sr=sr//f
    N,HOP=1024,512; nF=max(8,(len(y)-N)//HOP); hopS=HOP/sr
    win=0.5-0.5*np.cos(2*np.pi*np.arange(N)/N)
    odf=np.zeros(nF); prev=np.zeros(N//2)
    for fr in range(nF):
        s=fr*HOP; fftv=np.abs(np.fft.rfft(y[s:s+N]*win)[:N//2])
        m=np.log(1+fftv); d=m-prev; odf[fr]=np.maximum(0,d).sum(); prev=m
    Wm=max(2,round(0.35/hopS))
    def realca(src):
        out=np.zeros(len(src)); acc=0; fila=[]
        for i in range(len(src)):
            fila.append(src[i]); acc+=src[i]
            if len(fila)>Wm: acc-=fila.pop(0)
            out[i]=max(0,src[i]-acc/len(fila))
        return out
    odf2=realca(odf); meanO=odf2.mean() if nF else 0
    def acorrE(L):
        L=int(round(L))
        if L<2 or L>=nF-2: return 0
        a=odf2[:nF-L]; b=odf2[L:]
        return float((a*b).mean()) if len(a) else 0
    def combE(b):
        L=(60/b)/hopS
        return acorrE(L)+0.7*acorrE(L*2)+0.5*acorrE(L*3)+0.35*acorrE(L*4)
    curve=[(b,combE(b)) for b in np.arange(40,220.5,0.5)]
    cands=set()
    for i in range(2,len(curve)-2):
        v=curve[i][1]
        if v>curve[i-1][1] and v>=curve[i+1][1] and v>curve[i-2][1] and v>=curve[i+2][1]:
            cands.add(curve[i][0])
    if not cands: cands.add(100)
    for b in list(cands):
        for v in [b/2,b*2,b*3/4,b*4/3]:
            if 40<=v<=220: cands.add(round(v*2)/2)
    def beatScore(b):
        P=(60/b)/hopS; best=0; bph=0
        for s in range(24):
            ph=s*P/24; idx=np.arange(ph,nF,P).round().astype(int); idx=idx[idx<nF]
            v=odf2[idx].mean() if len(idx) else 0
            if v>best: best=v; bph=ph
        return best/(meanO+1e-9),bph
    def prefLento(b): return 1+0.12*np.exp(-0.5*((b-110)/70)**2)
    bestB=100; bestV=-1; bestPh=0
    for b in cands:
        sc,ph=beatScore(b); v=sc*prefLento(b)
        if v>bestV: bestV=v; bestB=b; bestPh=ph
    for b in np.arange(bestB-1.5,bestB+1.55,0.05):
        if b<40 or b>220: continue
        sc,ph=beatScore(b); v=sc*prefLento(b)
        if v>bestV: bestV=v; bestB=b; bestPh=ph
    dobro=bestB*2
    if dobro<=210:
        scD,phD=beatScore(dobro); scB,phB=beatScore(bestB)
        if scD>=scB*0.88: bestB=dobro; bestPh=phD
    return round(bestB*100)/100, round(max(0,bestPh*hopS),3)

# ---- localiza voz e musica ----
voz_ref=None
for p in ['/content/vozes/*(Vocals)*','/content/saida/*/*/vocals.*']:
    a=glob.glob(p)
    if a: voz_ref=a[0]; break
if not voz_ref: raise SystemExit('Voz nao encontrada. Rode as celulas 3 e 4.')
orig=glob.glob('/content/entrada/*')[0]
try: arquivo
except NameError: arquivo=os.path.basename(orig)
base=os.path.splitext(arquivo)[0]
os.makedirs('/content/partitura',exist_ok=True)

# ---- BPM real + tom (armadura de clave) pela musica completa ----
print('Detectando BPM e tom...')
yc,sr=librosa.load(orig,sr=22050,mono=True)
bpm,fase=detectar_bpm_orivox(yc,sr); batida=60.0/bpm
cm=librosa.feature.chroma_cqt(y=yc,sr=sr).mean(axis=1)
MA=np.array([6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88])
MI=np.array([6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17])
best=(-1,None,None)
for i in range(12):
    for perfil,tp in [(MA,'maior'),(MI,'menor')]:
        c=np.corrcoef(cm,np.roll(perfil,i))[0,1]
        if c>best[0]: best=(c,i,tp)
tonica_i,tipo=best[1],best[2]
graus=[0,2,4,5,7,9,11] if tipo=='maior' else [0,2,3,5,7,8,10]
escala=set((tonica_i+g)%12 for g in graus)
print(f'  BPM: {bpm} | Tom: {NOTAS[tonica_i]} {tipo}')

# ---- extrai a voz (pyin) ----
print('Extraindo a melodia da voz (pyin)... pode levar 1-2 min.')
yv,sr=librosa.load(voz_ref,sr=22050,mono=True)
f0,vf,vp=librosa.pyin(yv,fmin=150,fmax=500,sr=sr,hop_length=256)
tempos=librosa.times_like(f0,sr=sr,hop_length=256)
notas=[]; na=None; ini=0
for k in range(len(f0)):
    hz=f0[k]; m=int(round(librosa.hz_to_midi(hz))) if (hz and not np.isnan(hz)) else None
    if m!=na:
        if na is not None and (tempos[k]-ini)>0.04: notas.append([na,ini,tempos[k],70])
        na=m; ini=tempos[k]

# ---- consolida (junta fragmentacao de vibrato) ----
def consolidar(ns, sens=0.6):
    inv=1-sens; gapTol=0.10+inv*0.14; tolFirme=0.5; tolAmpla=2
    durBreve=0.12+inv*0.06; durIntruso=0.09
    arr=sorted([list(n) for n in ns],key=lambda x:(x[1],x[0]))
    def perto(i):
        n=arr[i]
        for j in range(max(0,i-3),min(len(arr),i+4)):
            if j==i: continue
            c=arr[j]
            if abs(c[0]-n[0])<=tolAmpla and (n[1]-c[2])<gapTol and (c[1]-n[2])<gapTol: return True
        return False
    arr=[n for i,n in enumerate(arr) if (n[2]-n[1])>durIntruso or perto(i)]
    out=[]
    for n in arr:
        durN=n[2]-n[1]; alvo=None
        for i in range(len(out)-1,-1,-1):
            if len(out)-i>6: break
            c=out[i]
            if n[1]-c[2]>gapTol: continue
            d=abs(n[0]-c[0])
            if d<=tolFirme:
                if (n[1]-c[2])<=0.10: alvo=c; break
                else: continue
            if d<=tolAmpla and durN<=durBreve: alvo=c; break
        if alvo:
            dA=alvo[2]-alvo[1]
            if durN>dA: alvo[0]=n[0]
            alvo[2]=max(alvo[2],n[2])
        else: out.append([n[0],n[1],n[2],n[3] or 70])
    return sorted(out,key=lambda x:x[1])
cons=consolidar(notas)

# ---- corrige alturas para dentro do tom ----
def corrige(m):
    if NOTAS[m%12] in escala: return m
    for d in [1,-1,2,-2]:
        if NOTAS[(m+d)%12] in escala: return m+d
    return m
for n in cons: n[0]=corrige(int(n[0]))

# ---- ancora no primeiro canto ----
rms=librosa.feature.rms(y=yv,frame_length=1024,hop_length=256)[0]
trms=librosa.frames_to_time(np.arange(len(rms)),sr=sr,hop_length=256)
prim=trms[np.argmax(rms>rms.max()*0.12)]
inicio=prim-batida  # voz entra na batida 2 do compasso 1

# ---- legato leve (engole micro-silencios < 0.5 batida) ----
cons.sort(key=lambda x:x[1])
for i in range(len(cons)-1):
    if cons[i+1][1]-cons[i][2] < batida*0.5: cons[i][2]=cons[i+1][1]

# ---- monta a partitura com figuras limpas + divisao por compasso ----
from music21 import stream, note as m21note, tempo as m21tempo, meter, key, duration, metadata, tie
passo=0.25
eventos=[]
for m,s0,s1,v in cons:
    off=round(((s0-inicio)/batida)/passo)*passo
    dur=round(((s1-s0)/batida)/passo)*passo
    if off<0 or dur<passo: continue
    eventos.append([off,max(passo,dur),int(m)])
FIG=[0.25,0.5,0.75,1.0,1.5,2.0,3.0,4.0]
def eh_simples(d): return any(abs(d-f)<0.01 for f in FIG)
def fig(d):
    for f in FIG:
        if abs(d-f)<0.01: return f
    return min([4.0,2.0,1.0,0.5,0.25],key=lambda f:abs(f-d))
def dividir(off,dur,midi):
    fim=off+dur; cde=int(off//4); cate=int((fim-1e-6)//4)
    if eh_simples(dur) and cde==cate: return [[off,dur,midi]]
    partes=[]; atual=off
    while atual<fim-1e-6:
        pc=(int(atual//4)+1)*4; seg=min(fim,pc); d=seg-atual
        if eh_simples(d): partes.append([atual,d,midi])
        else:
            a2=atual
            while a2<seg-1e-6:
                pt=np.floor(a2+1e-6)+1; s2=min(seg,pt)
                partes.append([a2,s2-a2,midi]); a2=s2
        atual=seg
    return partes
s=stream.Part()
s.insert(0,m21tempo.MetronomeMark(number=bpm))
km='minor' if tipo=='menor' else 'major'
s.insert(0,key.Key(NOTAS[tonica_i].replace('#','#').lower() if tipo=='menor' else NOTAS[tonica_i], km))
s.insert(0,meter.TimeSignature('4/4'))
for off,dur,midi in eventos:
    partes=dividir(off,dur,midi)
    for i,(o,d,mi) in enumerate(partes):
        n=m21note.Note(mi); n.duration=duration.Duration(fig(d))
        if len(partes)>1:
            n.tie=tie.Tie('start' if i==0 else ('stop' if i==len(partes)-1 else 'continue'))
        s.insert(o,n)
s_m=s.makeMeasures(); s_m.makeTies(inPlace=True)
s_m.insert(0,metadata.Metadata()); s_m.metadata.title=base
s_m.metadata.composer='Transcricao Orivox'

nf=list(s_m.recurse().notes)
print(f'Partitura: {len(s_m.getElementsByClass("Measure"))} compassos, {len(nf)} notas, tom {NOTAS[tonica_i]} {tipo}')

voz_xml=f'/content/partitura/{base}_partitura.musicxml'
s_m.write('musicxml',fp=voz_xml)
voz_mid=f'/content/partitura/{base}_partitura.mid'
s_m.write('midi',fp=voz_mid)
print('Partitura gerada (MusicXML + MIDI). Abra no MuseScore/Encore.')
for f in [voz_xml,voz_mid]: files.download(f)


## Celula 8 - EXTRA: Cifra dos acordes (para a aba Acordes do Orivox)

Detecta a progressao de acordes da musica com o tempo de cada mudanca.
Gera um TXT legivel e um MusicXML para importar no filtro harmonico do Orivox.
*Opcional. Roda por ultimo e instala suas proprias ferramentas.*

In [ ]:
# Cifra dos acordes SEM TensorFlow (cromagrama CQT + templates). Gera TXT e PDF.
import subprocess
subprocess.run(['pip','install','-q','librosa','soundfile','scipy','reportlab'], check=False)

import glob, os
import numpy as np
import librosa
from scipy.ndimage import uniform_filter1d
from google.colab import files

caminho = glob.glob('/content/entrada/*')[0]
base = os.path.splitext(os.path.basename(caminho))[0]
os.makedirs('/content/cifra', exist_ok=True)

print('Detectando acordes (cromagrama)... 1 a 2 minutos.')
y, sr = librosa.load(caminho, sr=22050, mono=True)
# tuning corrige desafinacao; harmonico realca as notas da harmonia
y_harm = librosa.effects.harmonic(y, margin=3)
chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr, hop_length=2048)
chroma = uniform_filter1d(chroma, size=8, axis=1)

NOTAS = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
def _tmpl(root, tipo):
    t = np.zeros(12); ints = [0,4,7] if tipo=='maj' else [0,3,7]
    for i in ints: t[(root+i)%12] = 1
    return t/np.linalg.norm(t)
templates = []
for r in range(12):
    templates.append((NOTAS[r], _tmpl(r,'maj')))
    templates.append((NOTAS[r]+'m', _tmpl(r,'min')))

tempos = librosa.frames_to_time(np.arange(chroma.shape[1]), sr=sr, hop_length=2048)
seq = []
for j in range(chroma.shape[1]):
    v = chroma[:,j]; n = np.linalg.norm(v)
    if n < 0.02: seq.append('N'); continue
    v = v/n; melhor, score = 'N', -1
    for nome, t in templates:
        s = float(np.dot(v, t))
        if s > score: score, melhor = s, nome
    # so aceita se a correlacao for razoavel (evita ruido virar acorde)
    seq.append(melhor if score > 0.55 else 'N')

acordes = []
i = 0
while i < len(seq):
    nome = seq[i]
    if nome == 'N': i += 1; continue
    j = i
    while j < len(seq) and seq[j] == nome: j += 1
    if (j-i) >= 4 and (not acordes or acordes[-1][1] != nome):
        acordes.append((float(tempos[i]), nome))
    i = j
print(f'Detectados {len(acordes)} blocos de acorde.')

gerou = []
def mmss(seg):
    m = int(seg // 60); s = int(seg % 60); return f'{m}:{s:02d}'

# 1) TXT legivel
txt = f'Cifra de {base}\n(gerada pelo Separador Orivox - confira com o ouvido)\n\n'
txt += '  '.join(f'{mmss(t)} {nome}' for t, nome in acordes) if acordes else '(nenhum acorde detectado - a musica pode ter harmonia complexa)'
acordes_txt = f'/content/cifra/{base}_acordes.txt'
open(acordes_txt, 'w').write(txt)
gerou.append(acordes_txt)
print('Cifra em texto:', os.path.basename(acordes_txt))

# 2) PDF legivel (uma cifra de verdade, acordes em grade com o tempo)
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.units import cm
    from reportlab.pdfgen import canvas
    pdf_path = f'/content/cifra/{base}_cifra.pdf'
    c = canvas.Canvas(pdf_path, pagesize=A4)
    w, h = A4
    c.setFont('Helvetica-Bold', 18); c.drawString(2*cm, h-2*cm, 'Cifra - '+base[:40])
    c.setFont('Helvetica', 9); c.drawString(2*cm, h-2.6*cm, 'gerada pelo Separador Orivox - confira com o ouvido')
    y0 = h-4*cm; x = 2*cm; porLinha = 4
    if acordes:
        for i,(t,nome) in enumerate(acordes):
            c.setFont('Helvetica-Bold', 15); c.drawString(x, y0, nome)
            c.setFont('Helvetica', 7); c.drawString(x, y0-0.4*cm, mmss(t))
            x += 4.2*cm
            if (i+1) % porLinha == 0:
                x = 2*cm; y0 -= 1.5*cm
                if y0 < 2*cm: c.showPage(); y0 = h-2*cm
    else:
        c.setFont('Helvetica', 12); c.drawString(2*cm, y0, 'Nenhum acorde detectado nesta musica.')
    c.save()
    gerou.append(pdf_path)
    print('Cifra em PDF:', os.path.basename(pdf_path))
except Exception as e:
    print('PDF nao gerado (o TXT ja serve):', e)

for f in gerou:
    files.download(f)